In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
duration = 0.5
dt = 1e-4
t = np.arange(0, duration, dt)

I_amp = 2e-9  # try 2 nA
I = np.zeros_like(t)

I[(t >= 0.1) & (t <= 0.4)] = I_amp

def integrate_and_fire_current_pulse(
    I,
    t,
    dt=1e-4,
    V_rest=-70e-3,
    V_reset=-80e-3,
    V_threshold=-54e-3,
    R_m=10e6,
    tau_m=10e-3
):
    V = np.zeros_like(t)
    V[0] = V_rest

    spike_times = []

    for i in range(len(t) - 1):

        dVdt = (-(V[i] - V_rest) + R_m * I[i]) / tau_m
        V[i+1] = V[i] + dt * dVdt

        if V[i+1] >= V_threshold:
            spike_times.append(t[i+1])
            V[i+1] = V_reset

    return V, np.array(spike_times)

V, spike_times = integrate_and_fire_current_pulse(I, t)

pulse_start = 0.1
pulse_end = 0.4
pulse_duration = pulse_end - pulse_start

spikes_during_pulse = spike_times[
    (spike_times >= pulse_start) & (spike_times <= pulse_end)
]

firing_rate = len(spikes_during_pulse) / pulse_duration

currents = np.linspace(0, 5e-9, 50)
rates = []

for I_amp in currents:
    I = np.zeros_like(t)
    I[(t >= 0.1) & (t <= 0.4)] = I_amp

    V, spike_times = integrate_and_fire_current_pulse(I, t)

    spikes_during_pulse = spike_times[
        (spike_times >= 0.1) & (spike_times <= 0.4)
    ]

    rate = len(spikes_during_pulse) / 0.3
    rates.append(rate)

plt.plot(currents * 1e9, rates, marker='o')
plt.xlabel("Input current (nA)")
plt.ylabel("Firing rate (Hz)")
plt.title("F-I curve")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(t * 1000, V * 1000)
plt.xlabel("Time (ms)")
plt.ylabel("Membrane potential (mV)")
plt.title("Leaky integrate-and-fire neuron")
plt.show()

In [ ]:
def IandF(I, t, dt=1e-4):
    V_rest = -70e-3
    V_reset = -80e-3
    V_threshold = -54e-3
    R_m = 10e6
    tau_m = 10e-3

    V = np.zeros_like(t)
    V[0] = V_rest
    spike_times = []

    for i in range(len(t) - 1):

        dVdt = (-(V[i] - V_rest) + R_m * I[i]) / tau_m
        V[i+1] = V[i] + dt * dVdt

        if V[i+1] >= V_threshold:
            spike_times.append(t[i+1])
            V[i+1] = V_reset


    return V, spike_times


t = np.arange(0, 1, 1e-4)
I = np.zeros_like(t)
I[(t>=0.1) & (t<=0.4)] = 2e-9  # 2 nA pulse from 100 ms to 400 ms

V, spike_times = IandF(I, t)

I_array = np.linspace(0, 5e-9, 50)
firing_rates = []

for I_amp in I_array:
    I = np.zeros_like(t)
    I[(t >= 0.1) & (t <= 0.4)] = I_amp

    V, spike_times = IandF(I, t)


    rate = len(spike_times) / 0.3
    firing_rates.append(rate)

plt.figure(figsize=(12, 5))
plt.subplot(1,2,1)
plt.plot(t, V * 1e3)
plt.xlabel('Time (s)')
plt.ylabel('Membrane Potential (mV)')
plt.title('Integrate-and-Fire Neuron')
plt.plot(spike_times, [0] * len(spike_times), 'ro', label='Spikes')
plt.legend()




plt.subplot(1,2,2)
plt.plot(I_array * 1e9, firing_rates, marker='o')
plt.xlabel('Input Current (nA)')
plt.ylabel('Firing Rate (Hz)')
plt.title('F-I Curve for Integrate-and-Fire Neuron')
#plt.savefig('IandF_FI_curve.png')
plt.show()

In [ ]:
def lif_theory_rate(I, R_m=10e6, tau_m=10e-3,
                    V_rest=-70e-3, V_reset=-80e-3, V_threshold=-54e-3):

    I = np.array(I)
    rates = np.zeros_like(I)

    valid = R_m * I + V_rest > V_threshold

    rates[valid] = 1 / (
        tau_m * np.log(
            (R_m * I[valid] + V_rest - V_reset) /
            (R_m * I[valid] + V_rest - V_threshold)
        )
    )

    return rates

theory_rates = lif_theory_rate(I_array)

plt.plot(I_array * 1e9, firing_rates, 'o', label="simulation")
plt.plot(I_array * 1e9, theory_rates, label="theory")
plt.xlabel("Input current (nA)")
plt.ylabel("Firing rate (Hz)")
plt.legend()
plt.show()

Rate adaptation included

In [ ]:
def IandF_with_adaptation(I, t, dt = 1e-4):
    V_rest = -65e-3
    V_reset = -65e-3
    V_threshold = -50e-3
    R_m = 90e6
    tau_m = 30e-3
    tau_a = 100e-3
    delta_a = 0.06
    E_a = -70e-3

    V = np.zeros_like(t)
    V[0] = V_rest

    spike_times = []
    a = [0]

    for i in range(len(t) - 1):

        dVdt = (-(V[i] - V_rest) + R_m * I[i] - a[-1] * (V[i] - E_a)) / tau_m
        V[i+1] = V[i] + dt * dVdt

        if V[i+1] >= V_threshold:
            spike_times.append(t[i+1])
            V[i+1] = V_reset
            a.append(a[-1] + delta_a)

        da_dt = -a[-1] / tau_a
        a.append(a[-1] + da_dt * dt)

    return V, spike_times, a

t = np.arange(0, 1, 1e-4)

firing_rates_adapt = []
I_array = np.linspace(0, 1e-9, 15)
for I_amp in I_array:
    I = np.zeros_like(t)
    I[(t >= 0.1) & (t <= 0.4)] = I_amp

    V, spike_times, a = IandF_with_adaptation(I, t)

    rate = len(spike_times) / 0.3
    firing_rates_adapt.append(rate)


isi = np.diff(spike_times)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(isi * 1000, marker='o')
plt.xlabel('Spike Index')
plt.ylabel('Interspike Interval (ms)')
plt.title('Interspike Intervals with Adaptation')

plt.subplot(1, 2, 2)
plt.plot(I_array * 1e9, firing_rates_adapt, marker='o')
plt.xlabel('Input Current (nA)')
plt.ylabel('Firing Rate (Hz)')
plt.title('F-I Curve for Integrate-and-Fire Neuron with Adaptation')
plt.show()


Presynaptic spike

In [ ]:
def presyn_spike(I, t, t_spikes, dt = 1e-4, V_rest=-70e-3, V_reset=-80e-3, V_threshold=-54e-3, R_m=10e6, tau_m=10e-3, E_s =0, tau_s = 10e-3):
    rmgs = 0.5
    Pmax = 1
    V = np.zeros_like(t)
    V[0] = V_rest
    z = [0]
    Ps = [0]
    spike_times = []
    syn_current = [0]

    for i in range(len(t) - 1):
        if i*dt in t_spikes:
            z.append(1)
        else:
            z.append(z[-1] - z[-1] * dt / tau_s)

        dPsdt = (np.e*Pmax*z[-1] - Ps[-1])/tau_s
        Ps.append(Ps[-1] + dPsdt * dt)

        dVdt = (-(V[i] - V_rest) + R_m * I[i] + rmgs * Ps[-1] * (E_s - V[i])) / tau_m   
        V[i+1] = V[i] + dVdt * dt
        if V[i+1] >= V_threshold:
            spike_times.append(t[i+1])
            V[i+1] = V_reset
        syn_current.append(rmgs * Ps[-1] * (E_s - V[i]))

    return V, spike_times, syn_current, Ps

t = np.arange(0, 0.5, 1e-4)
t_spikes = [50e-3, 150e-3, 190e-3, 300e-3, 320e-3, 400e-3, 410e-3]

I = np.zeros_like(t)

V, spike_times, syn_current, Ps = presyn_spike(I, t, t_spikes)


plt.plot(t, V * 1e3)
plt.xlabel('Time (s)')
plt.ylabel('Membrane Potential (mV)')
plt.title('Integrate-and-Fire Neuron with Presynaptic Spikes')

plt.plot(spike_times, [0] * len(spike_times), 'ro', label='Presynaptic spikes')
plt.legend()

plt.show()



In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(t, syn_current, label='Synaptic Current')
plt.xlabel('Time (s)')
plt.ylabel('Synaptic Current (A)')
plt.title('Synaptic Current from Presynaptic Spikes')
plt.legend()

plt.subplot(1,2,2)
plt.plot(t, Ps, label='Synaptic Conductance (Ps)')
plt.xlabel('Time (s)')
plt.ylabel('Synaptic Conductance (S)')
plt.axhline(1, color='red', linestyle='--', label='Reference Line')
plt.title('Synaptic Conductance from Presynaptic Spikes')
plt.legend()
plt.show()

In [ ]:
def presyn_spike(I, t, t_spikes, dt = 1e-4, V_rest=-70e-3, V_reset=-80e-3, V_threshold=-54e-3, R_m=10e6, tau_m=10e-3, E_s =0, tau_s = 10e-3):
    rmgs = 0.5
    Pmax = 0.5
    V = np.zeros_like(t)
    V[0] = V_rest
    z = [0]
    Ps = [0]
    spike_times = []
    syn_current = [0]
    spike_indices = set((t_spikes / dt).astype(int))
    for i in range(len(t) - 1):
        if i in spike_indices:
            z.append(1)
        else:
            z.append(z[-1] - z[-1] * dt / tau_s)

        dPsdt = (np.e*Pmax*z[-1] - Ps[-1])/tau_s
        Ps.append(Ps[-1] + dPsdt * dt)

        dVdt = (-(V[i] - V_rest) + R_m * I[i] + rmgs * Ps[-1] * (E_s - V[i])) / tau_m   
        V[i+1] = V[i] + dVdt * dt
        if V[i+1] >= V_threshold:
            spike_times.append(t[i+1])
            V[i+1] = V_reset
        syn_current.append(rmgs * Ps[-1] * (E_s - V[i]))

    return V, spike_times, syn_current, Ps


def presyn_spike_upperbound(I, t, t_spikes, dt = 1e-4, V_rest=-70e-3, V_reset=-80e-3, V_threshold=-54e-3, R_m=10e6, tau_m=10e-3, E_s =0, tau_s = 10e-3):
    rmgs = 0.5
    Pmax = 0.5
    V = np.zeros_like(t)
    V[0] = V_rest
    z = [0]
    Ps = [0]
    spike_times = []
    syn_current = [0]
    spike_indices = set((t_spikes / dt).astype(int))

    for i in range(len(t) - 1):
        if i in spike_indices:
            z.append(1)
        else:
            z.append(z[-1] - z[-1] * dt / tau_s)

        dPsdt = (np.e*Pmax*z[-1]*(1-Ps[-1]) - Ps[-1])/tau_s
        Ps.append(Ps[-1] + dPsdt * dt)

        dVdt = (-(V[i] - V_rest) + R_m * I[i] + rmgs * Ps[-1] * (E_s - V[i])) / tau_m   
        V[i+1] = V[i] + dVdt * dt
        if V[i+1] >= V_threshold:
            spike_times.append(t[i+1])
            V[i+1] = V_reset
        syn_current.append(rmgs * Ps[-1] * (E_s - V[i]))

    return V, spike_times, syn_current, Ps


duration = 1

t = np.arange(0, duration, 1e-4)

f = np.linspace(1, 100, 20)
Ps_old = []
Ps_new = []
for i, di in enumerate(f):
    t_spikes = np.arange(0.05, duration, 1/di)
    I = np.zeros_like(t)

    V1, spike_times1, syn_current1, Ps1 = presyn_spike(I, t, t_spikes)
    V2, spike_times2, syn_current2, Ps2 = presyn_spike_upperbound(I, t, t_spikes)

    transient = int(0.5/ 1e-4) # 0.5 seconds transient
    Ps_old.append(np.mean(Ps1[transient:]))
    Ps_new.append(np.mean(Ps2[transient:]))

plt.plot(f, Ps_old, label='Ps Mean (Original)', marker='o')
plt.plot(f, Ps_new, label='Ps Mean (Upper Bound)', marker='s')
plt.axhline(0.5, color='red', linestyle='--', label='Pmax = 0.5')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Ps Mean')
plt.legend()
plt.show()

Coupled I and F model 

In [ ]:
def coupled_IandF_network(t, dt = 1e-4, state=False):
    V_rest = -70e-3 
    V_th = -54e-3
    V_reset = -80e-3
    RmIe = 18e-3
    tau_m = 20e-3
    rmgs = 0.15
    tau_s = 10e-3
    Pmax = 0.5

    V1 = np.zeros_like(t)
    V2 = np.zeros_like(t)
    V1[0] = V_rest
    V2[0] = V_rest - 5e-3

    z12 = [0]
    z21 = [0]
    Ps12 = [0]
    Ps21 = [0]

    spike_times1 = []
    spike_times2 = []

    for i in range(len(t) - 1):
        dPs12_dt = (np.e * Pmax * z12[-1] * (1 - Ps12[-1]) - Ps12[-1]) / tau_s
        dPs21_dt = (np.e * Pmax * z21[-1] * (1 - Ps21[-1]) - Ps21[-1]) / tau_s

        Ps12.append(Ps12[-1] + dPs12_dt * dt)
        Ps21.append(Ps21[-1] + dPs21_dt * dt)
        if state == False:
            E_s = -80e-3
        else:
            E_s = 0
        
        dV1_dt = (-(V1[i] - V_rest) + RmIe + rmgs * Ps21[-1-1] * (E_s - V1[i])) / tau_m
        dV2_dt = (-(V2[i] - V_rest) + RmIe + rmgs * Ps12[-1-1] * (E_s - V2[i])) / tau_m

        V1[i+1] = V1[i] + dV1_dt * dt
        V2[i+1] = V2[i] + dV2_dt * dt

        if V1[i+1] >= V_th:
            spike_times1.append(t[i+1])
            V1[i+1] = V_reset
            z12.append(1)
        else:
            z12.append(z12[-1] - z12[-1] * dt / tau_s)

        if V2[i+1] >= V_th:
            spike_times2.append(t[i+1])
            V2[i+1] = V_reset
            z21.append(1)
        else:
            z21.append(z21[-1] - z21[-1] * dt / tau_s)

    
    return V1, V2, spike_times1, spike_times2, Ps12, Ps21


t = np.arange(0, 6, 1e-4)
V1, V2, spike_times1, spike_times2, Ps12, Ps21 = coupled_IandF_network(t, state=True)

mask = t > 5

plt.plot(t[mask], V1[mask] * 1e3, label='Neuron 1')
plt.plot(t[mask], V2[mask] * 1e3, label='Neuron 2')
plt.xlabel('Time (s)')
plt.ylabel('Membrane Potential (mV)')
plt.title('Coupled Integrate-and-Fire Neurons')
plt.legend()
plt.show()



In [ ]:
mask1 = np.array(spike_times1) > 5
mask2 = np.array(spike_times2) > 5

plt.eventplot(
    [np.array(spike_times1)[mask1], np.array(spike_times2)[mask2]],
    lineoffsets=[1, 2], colors=['blue', 'orange']
)
plt.yticks([1, 2], ["Neuron 1", "Neuron 2"])
plt.title("Spike Raster Plot of Coupled Neurons - Inhibitory")
plt.xlabel("Time (s)")
plt.show()

mask1 = np.array(spike_times1) > 5
mask2 = np.array(spike_times2) > 5

plt.eventplot(
    [np.array(spike_times1)[mask1], np.array(spike_times2)[mask2]],
    lineoffsets=[1, 2], colors=['blue', 'orange']
)
plt.yticks([1, 2], ["Neuron 1", "Neuron 2"])
plt.title("Spike Raster Plot of Coupled Neurons - Excitatory")
plt.xlabel("Time (s)")
plt.show()

sp1 = np.array(spike_times1)
sp2 = np.array(spike_times2)

# only last second
sp1 = sp1[sp1 > 5]
sp2 = sp2[sp2 > 5]

n = min(len(sp1), len(sp2))
phase_delay = sp2[:n] - sp1[:n]

print("Excitatory coupling phase delay:")
print(np.mean(phase_delay) * 1000, "ms")

print("Neuron 1 period:", np.mean(np.diff(sp1)) * 1000, "ms")
print("Neuron 2 period:", np.mean(np.diff(sp2)) * 1000, "ms")